In [1]:
import nltk
import re
from gensim import corpora, models
from gensim.models import LdaModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## Sample Text Corpus

In [3]:
documents = [
    "Apple launched a new iPhone with an improved camera and longer battery life.",
    "The stock market rallied today as technology shares surged on strong earnings.",
    "The new basketball season starts in October with exciting matchups.",
    "Google released an update to Android with better privacy features.",
    "The Federal Reserve announced interest rates will remain steady this quarter.",
    "The Lakers will face the Celtics in the opening game of the NBA season.",
    "Tesla unveiled a new electric vehicle with advanced autopilot features.",
    "Investors are worried about inflation as bond yields climbed higher today.",
    "The soccer world cup qualifiers begin next month with several key matches.",
    "Microsoft launched a new Surface laptop with improved performance and display."
]

## Text Preprocessing

In [4]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words and len(t) > 3]
    return tokens

processed_docs = [preprocess(doc) for doc in documents]
print("Processed documents:")
for i, doc in enumerate(processed_docs, 1):
    print(f"Doc {i}: {doc}")

Processed documents:
Doc 1: ['apple', 'launched', 'iphone', 'improved', 'camera', 'longer', 'battery', 'life']
Doc 2: ['stock', 'market', 'rallied', 'today', 'technology', 'shares', 'surged', 'strong', 'earnings']
Doc 3: ['basketball', 'season', 'starts', 'october', 'exciting', 'matchups']
Doc 4: ['google', 'released', 'update', 'android', 'better', 'privacy', 'features']
Doc 5: ['federal', 'reserve', 'announced', 'interest', 'rates', 'remain', 'steady', 'quarter']
Doc 6: ['lakers', 'face', 'celtics', 'opening', 'game', 'season']
Doc 7: ['tesla', 'unveiled', 'electric', 'vehicle', 'advanced', 'autopilot', 'features']
Doc 8: ['investors', 'worried', 'inflation', 'bond', 'yields', 'climbed', 'higher', 'today']
Doc 9: ['soccer', 'world', 'qualifiers', 'begin', 'next', 'month', 'several', 'matches']
Doc 10: ['microsoft', 'launched', 'surface', 'laptop', 'improved', 'performance', 'display']


## Build Dictionary & Bag-of-Words Corpus

In [5]:
dictionary = corpora.Dictionary(processed_docs)
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"Dictionary size: {len(dictionary)}")
print(f"Number of documents: {len(bow_corpus)}")
print("\nSample BoW for Doc 1:")
print(bow_corpus[0])

Dictionary size: 69
Number of documents: 10

Sample BoW for Doc 1:
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1)]


## Train LDA Model

In [6]:
lda_model = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=20,
    alpha='auto',
    per_word_topics=False
)

print(f"LDA model trained with {lda_model.num_topics} topics.")

LDA model trained with 3 topics.


## Inspect Topics — Top Words

In [7]:
topics = lda_model.print_topics(num_words=10)
for topic_id, words in topics:
    print(f"Topic {topic_id}: {words}\n")

Topic 0: 0.026*"features" + 0.026*"update" + 0.026*"android" + 0.026*"season" + 0.026*"google" + 0.026*"lakers" + 0.026*"game" + 0.026*"better" + 0.026*"released" + 0.026*"opening"

Topic 1: 0.030*"improved" + 0.030*"launched" + 0.030*"season" + 0.030*"qualifiers" + 0.030*"several" + 0.030*"begin" + 0.030*"matches" + 0.030*"laptop" + 0.030*"soccer" + 0.030*"next"

Topic 2: 0.050*"today" + 0.028*"market" + 0.028*"surged" + 0.028*"strong" + 0.028*"investors" + 0.028*"worried" + 0.028*"earnings" + 0.028*"technology" + 0.028*"climbed" + 0.028*"rallied"



## Visualize Topic-Word Distributions

In [8]:
topic_words = []
for topic_id in range(lda_model.num_topics):
    words = lda_model.show_topic(topic_id, topn=8)
    topic_words.append([word for word, prob in words])

max_len = max(len(w) for w in topic_words)
for topic_id, words in enumerate(topic_words):
    padded = words + [""] * (max_len - len(words))
    row = f"Topic {topic_id:<3}: " + " | ".join(f"{w:<15}" for w in padded)
    print(row)

Topic 0  : features        | update          | android         | season          | google          | lakers          | game            | better         
Topic 1  : improved        | launched        | season          | qualifiers      | several         | begin           | matches         | laptop         
Topic 2  : today           | market          | surged          | strong          | investors       | worried         | earnings        | technology     


## Assign Topics to Documents

In [9]:
for i, bow in enumerate(bow_corpus):
    doc_topics = lda_model.get_document_topics(bow, minimum_probability=0.0)
    dominant = max(doc_topics, key=lambda x: x[1])
    print(
        f"Doc {i+1:<3}: dominant_topic={dominant[0]}  "
        f"prob={dominant[1]:.4f}  "
        f"text={documents[i][:60]}..."
    )

Doc 1  : dominant_topic=0  prob=0.9866  text=Apple launched a new iPhone with an improved camera and long...
Doc 2  : dominant_topic=2  prob=0.9865  text=The stock market rallied today as technology shares surged o...
Doc 3  : dominant_topic=1  prob=0.9800  text=The new basketball season starts in October with exciting ma...
Doc 4  : dominant_topic=0  prob=0.9847  text=Google released an update to Android with better privacy fea...
Doc 5  : dominant_topic=0  prob=0.9866  text=The Federal Reserve announced interest rates will remain ste...
Doc 6  : dominant_topic=0  prob=0.9822  text=The Lakers will face the Celtics in the opening game of the ...
Doc 7  : dominant_topic=2  prob=0.9828  text=Tesla unveiled a new electric vehicle with advanced autopilo...
Doc 8  : dominant_topic=2  prob=0.9849  text=Investors are worried about inflation as bond yields climbed...
Doc 9  : dominant_topic=1  prob=0.9849  text=The soccer world cup qualifiers begin next month with severa...
Doc 10 : dominant_t